# AG_PRAXIS NB05 — Baseline Models

This is the first notebook that trains anything. Everything the rest of the project
produces is compared against what comes out of here, so what it has to establish is not
one number but a floor: what a published model scores on this dataset, what the same
model scores once the split stops handing it the recording, and what a model with no
neural architecture at all scores on the same rows.

Three models, and the differences between them are the point.

The first is the convolutional network published by Mohammadi et al. (2024), run on the
split the dataset was distributed with, with its own preprocessing, as published. The
second is that same network, unchanged in every respect, on the two-tier split built in
NB04. Exactly one thing differs between the two, and it is the split. The third is a
random forest on single records, which is here so that at least one comparison point does
not depend on a neural architecture being the right idea in the first place.

The published reproduction runs all three of the groupings the dataset supports, nineteen
classes, six and two, because those are the numbers the paper reports and a reproduction
has to be comparable to the thing it reproduces. The other two models run nineteen and
six. Seven runs in total, and they are not run in the order they are described in: the
order is set later, by what matters most if the session ends early.

Every result is reported twice over: weighted, which is what the common classes did, and
macro, which is what the average class did. The largest class in this corpus holds two
thousand times as many rows as the smallest, so those two numbers answer different
questions, and the distance between them is what this notebook exists to make visible.

The data sits on Drive and the code sits in the repository, so the first block mounts one
and clones the other, and records the commit it is running from.

Training changes what a cell has to do before it is allowed to print. Every run below
writes its configuration, its predictions and the fitted model to disk as the last thing
its training statement does, and only then is anything displayed. A score printed by a run
that did not save cannot be checked afterwards, cannot be compared against the next run
and cannot go in the ledger, so it is worth nothing. That is also why fitting and saving
are a single statement everywhere in this notebook rather than two: a Colab session that
ends between two statements loses whatever the first one produced, and a full pass here is
hours long.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"
NOTEBOOK = "AG_PRAXIS_NB05_baseline_models.ipynb"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

The paths, the seed, the batch size and the number of epochs come from `config/base.yaml`.
What the data looks like comes from the two files NB04 wrote: `NB04_manifest.json`, which
says which columns survived and what the arrays hold, and `splits.json`, which says which
rows of which file are on each side of both splits. Nothing is retyped from either.

Two output folders again, `NB05_fast` and `NB05`, so a quick check of the pipeline cannot
overwrite a result.

In [ ]:
import gc
import json
import random
import textwrap
import time

import numpy as np
import pandas as pd

from baselines import mohammadi as mo
from src import inventory as inv
from src import runs as rn
from src.captures import group_six, group_two

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
BATCH_SIZE = int(CFG["training"]["batch_size"])
EPOCHS = int(CFG["training"]["epochs"])
TRAIN_DIR = Path(CFG["paths"]["train_dir"])
TEST_DIR = Path(CFG["paths"]["test_dir"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])
OUT_DIRS = {"fast": ARTIFACTS / "NB05_fast", "full": ARTIFACTS / "NB05"}
NB04_DIRS = {"fast": ARTIFACTS / "NB04_fast", "full": ARTIFACTS / "NB04"}

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(
        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "
        "config/base.yaml is wrong. Nothing this notebook writes would survive."
    )


def first_existing(candidates, what):
    found = next((Path(p) for p in candidates if Path(p).exists()), None)
    if found is None:
        raise FileNotFoundError(
            f"{what} not found. NB04 has to have been run and its output moved into "
            f"data/processed/. Looked in: {[str(p) for p in candidates]}"
        )
    return found


MANIFEST_PATH = first_existing(
    [
        REPO_ROOT / "data" / "processed" / "NB04_manifest.json",
        NB04_DIRS["full"] / "NB04_manifest.json",
    ],
    "NB04_manifest.json",
)
SPLITS_PATH = first_existing(
    [NB04_DIRS["full"] / "splits.json", REPO_ROOT / "data" / "processed" / "splits.json"],
    "splits.json",
)

MANIFEST = json.loads(MANIFEST_PATH.read_text())
SPLITS = json.loads(SPLITS_PATH.read_text())
if MANIFEST.get("is_fast_pass"):
    raise ValueError(f"{MANIFEST_PATH} came from NB04's fast pass and is not a result")

FEATURES = list(MANIFEST["columns"]["kept"])
CLASSES = sorted(MANIFEST["arrays"]["records_train"]["by_class"])

pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)

print(f"seed          : {SEED}")
print(f"batch size    : {BATCH_SIZE}")
print(f"epochs        : {EPOCHS}")
print(f"manifest      : {MANIFEST_PATH}")
print(f"  written by  : {MANIFEST['generated_by']} at {MANIFEST['git_sha']} on "
      f"{MANIFEST['generated_on']}")
print(f"splits        : {SPLITS_PATH}")
print(f"features      : {len(FEATURES)}, after {', '.join(MANIFEST['columns']['dropped'])} was "
      "dropped")
print(f"classes       : {len(CLASSES)}")
print(f"csv files     : {TRAIN_DIR} and {TEST_DIR}")
print(f"arrays        : {NB04_DIRS['full']}")
print(f"fast pass to  : {OUT_DIRS['fast']}")
print(f"full pass to  : {OUT_DIRS['full']}")
print()
print("the two splits, as NB04 recorded them")
for protocol, document in SPLITS.items():
    parts = ", ".join(
        f"{name} {counts:,}" for name, counts in document["rows_per_partition"].items()
    )
    print(f"  {protocol:<9} {document['n_blocks']:>3} blocks over {document['n_files']} files   "
          f"{parts}")

assert len(FEATURES) == 44, f"expected 44 features from the manifest, got {len(FEATURES)}"
assert len(CLASSES) == 19, f"expected 19 classes, got {len(CLASSES)}"
assert mo.FIT["batch_size"] == BATCH_SIZE, (
    f"the published model trains at batch {mo.FIT['batch_size']} and config/base.yaml says "
    f"{BATCH_SIZE}. The baseline is not changed to match; the config is."
)
assert mo.FIT["epochs"] == EPOCHS, (
    f"the published model trains for {mo.FIT['epochs']} epochs and config/base.yaml says "
    f"{EPOCHS}. The baseline is not changed to match; the config is."
)

The dataset supports three groupings of the same rows, and the paper reports all three.
Nineteen classes is every attack type separately. Six collapses them into the families the
benchmark defines, with the qualification that an MQTT flood counts as MQTT rather than as
a flood. Two is attack against benign.

The three are built here as index tables rather than as three copies of the labels: a row
carries one integer saying which of the nineteen classes it is, and each grouping is a
lookup from that integer to the class the grouping calls it. The rows themselves are
untouched, so nothing can be scored on a differently prepared version of the same data.

In [ ]:
TASKS = ["19-class", "6-class", "2-class"]
GROUPING = {"19-class": lambda label: label, "6-class": group_six, "2-class": group_two}

TASK_CLASSES = {task: sorted({GROUPING[task](c) for c in CLASSES}) for task in TASKS}
TASK_CODES = {
    task: np.searchsorted(
        np.asarray(TASK_CLASSES[task]), np.asarray([GROUPING[task](c) for c in CLASSES])
    ).astype("int64")
    for task in TASKS
}


def codes_for(task, codes19):
    """The nineteen-class codes of some rows, re-expressed in the classes of `task`."""
    return TASK_CODES[task][np.asarray(codes19).astype("int64")]


def labels_for(task, codes19):
    """The same thing as names rather than as positions."""
    return np.asarray(TASK_CLASSES[task], dtype=str)[codes_for(task, codes19)]


grouping_table = pd.DataFrame(
    {
        "class": CLASSES,
        "six": [GROUPING["6-class"](c) for c in CLASSES],
        "two": [GROUPING["2-class"](c) for c in CLASSES],
        "train_rows": [MANIFEST["arrays"]["records_train"]["by_class"][c] for c in CLASSES],
    }
)
print(grouping_table.to_string(index=False))
print()
for task in TASKS:
    print(f"{task:<9} {len(TASK_CLASSES[task]):>2} classes   {', '.join(TASK_CLASSES[task])}")
print()
counts = grouping_table["train_rows"]
print(f"largest class over smallest, 19-class : {counts.max() / counts.min():,.1f} : 1")

assert "UNMAPPED" not in TASK_CLASSES["6-class"], "a class did not map into the six"
assert len(TASK_CLASSES["6-class"]) == 6 and len(TASK_CLASSES["2-class"]) == 2
assert not isinstance(grouping_table["six"].dtype, pd.CategoricalDtype)

The seed is set before any model is constructed, which for this notebook means before the
first weight matrix is drawn, before the first shuffle of a training set, and before the
random forest picks its first feature.

It is set again immediately before each model is built, inside the function that builds it,
so that no code can run between seeding and construction and every run starts from the same
weights. What that gives is a repeatable initialisation and a repeatable shuffle. It does
not give bit-identical GPU arithmetic, because the order in which a reduction accumulates
on a GPU is not fixed and floating-point addition is not associative. Repeats across seeds
are NB08's job; here the seed makes one run reproducible enough to compare against another
run of the same thing.

In [ ]:
import keras
import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

GPUS = tf.config.list_physical_devices("GPU")

print(f"seeded with   : {SEED}")
print(f"tensorflow    : {tf.__version__}")
print(f"keras         : {keras.__version__}")
print(f"gpu           : {[d.name for d in GPUS] if GPUS else 'none, this will be slow'}")

Before any of it runs, what a run is.

A run is a configuration, and the configuration is the ledger entry. Each one names its
parent and overrides exactly one key of it, so when two runs score differently there is one
candidate for why. That is checked rather than intended: `assert_single_change` compares the
configuration against its parent and raises if more than one key differs, and it is called
before the model is built rather than after the fit, so a run that would confound two
changes never starts at all. The parent is named by identifier and the configurations are
all built together, before anything trains, so the chain does not depend on the order the
runs happen to be executed in.

Two configurations have no parent. The published network on the distributed split is where
one chain starts, and the random forest is where the other does, because a forest and a
convolutional network differ in everything and pretending otherwise would make the check
meaningless. Every other run below is one key away from one of those two.

A run leaves five files: `config.json`, `metrics.json`, `y_true.npy`, `y_pred.npy` and the
model, written by `save_run` as the last statement of the statement that fitted the model.
Training seconds and inference seconds are part of the metrics for every run, and `save_run`
refuses to write a run that does not have them.

`metrics.json` is also what makes a re-run cheap. It is written last, so a directory that
has one holds a run that finished, and the driver reads that run back off disk instead of
fitting it again. A session that dies four runs in and is started again picks up at the
fifth. Setting `RESUME` to False below turns that off and fits everything from scratch;
deleting a single run's folder re-runs that one.

In [ ]:
RESUME = True

ROWS_PER_FILE = {"fast": 2_000, "full": None}
FAST_ROW_CAPS = {"train": 50_000, "test": 20_000}
FOREST_TRAIN_ROWS = {"fast": 20_000, "full": 1_000_000}
FOREST = {"n_estimators": 100, "min_samples_leaf": 20, "max_features": "sqrt"}
K_FOLDS = 5
PREDICT_BATCH = 4096
FIT_VERBOSE = 2
PROGRESS_EVERY = 10


def section(title):
    print()
    print("-" * 100)
    print(title)
    print("-" * 100)


def banner(lines):
    print()
    print("#" * 100)
    for line in lines:
        print(f"#  {line[:95]:<96}#")
    print("#" * 100)


def merge(results, part):
    """Fold a step's return value into the run, keeping every run each step produced."""
    results["runs"] = results.get("runs", []) + list(part.pop("runs", []))
    results.update(part)
    return results


def reader_progress(what, every=PROGRESS_EVERY):
    started = time.time()

    def report_line(i, total, block, n_rows):
        if i % every and i != total:
            return
        print(f"  {what:<9} file {i:>3}/{total}   {block['file'][:44]:<44} {n_rows:>10,} rows"
              f"   {time.time() - started:6.1f}s")

    return report_line


def report(run):
    """The headline numbers of one run, then its full per-class table."""
    metrics, config = run["metrics"], run["config"]
    print(f"{config['run_id']}   {config['model']}, {config['task']}, {config['split']} split")
    print(f"  trained on {metrics['n_train']:,} rows, tested on {metrics['n_test']:,}")
    print(f"  accuracy            {metrics['accuracy']:.4f}   "
          f"(chance {metrics['chance_rate']:.4f}, always the largest class "
          f"{metrics['majority_class_rate']:.4f})")
    print(f"  weighted P / R / F1 {metrics['weighted_precision']:.4f} / "
          f"{metrics['weighted_recall']:.4f} / {metrics['weighted_f1']:.4f}")
    print(f"  macro    P / R / F1 {metrics['macro_precision']:.4f} / "
          f"{metrics['macro_recall']:.4f} / {metrics['macro_f1']:.4f}")
    print(f"  weighted F1 minus macro F1   {metrics['weighted_f1'] - metrics['macro_f1']:.4f}")
    print(f"  train {metrics['train_seconds']:,.1f}s, predict "
          f"{metrics['inference_seconds']:,.1f}s "
          f"({metrics['inference_rows_per_second']:,.0f} rows/s)")
    print(f"  saved to {run['run_dir']}")
    print()
    print(rn.per_class_frame(metrics).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print()


def banked(done, plan, i):
    """What is on disk now that run `i` of the plan has finished."""
    names = ", ".join(run["config"]["run_id"] for run in done)
    print(f"banked {i} of {len(plan)}")
    print(textwrap.fill(names, width=94, initial_indent="  ", subsequent_indent="  "))
    print(textwrap.fill(plan[i - 1]["banked"], width=94, initial_indent="  ",
                        subsequent_indent="  "))
    left = len(plan) - i
    print(f"  {left} run{'' if left == 1 else 's'} still to come."
          if left else "  Nothing left to run.")
    print()


print(f"resume              : {RESUME}")
print(f"forest              : {FOREST}, {K_FOLDS}-fold on the training partition")
print(f"forest training rows: fast {FOREST_TRAIN_ROWS['fast']:,}, "
      f"full up to {FOREST_TRAIN_ROWS['full']:,}")
print(f"rows per csv file   : fast {ROWS_PER_FILE['fast']:,}, full every row")

The first of the two sets of rows is read from the capture files.

This is the only notebook in the project that opens a CSV, and the reproduction is the
reason. NB04 turned every row into a scaled array using a scaler fitted on its own
training partition, and every later notebook reads those arrays. Handing them to a model
whose whole value is that it was produced the way the published one was would put this
project's preprocessing inside the reproduction, and then the comparison the reproduction
exists to provide would be a comparison with something that is not the published result.

So the baseline reads the distributed split's files itself, from the file lists and row
ranges in `splits.json`, and fits its own scaler on the training side of that split and
nothing else. That code lives under `baselines/mohammadi/`, separate from `src/`, and is
not shared with anything this project does on its own account.

One thing is held in common on purpose. Both sides use the same forty-four columns, which
is every column in the files except `Drate`. NB01 read all 8,775,013 rows and found `Drate`
holding one value in every one of them, so it separates nothing from anything, and a
constant column left in would sit in the input as an extra position for the convolution to
slide over. Giving both sides the same columns is what makes the split comparison about the
split and not about the width of the input.

In [ ]:
def load_shipped(fast: bool) -> dict:
    section("Reading the distributed split from the capture files")
    mode = "fast" if fast else "full"
    cap = ROWS_PER_FILE[mode]
    document = SPLITS["shipped"]

    blocks = {
        name: mo.attach_paths(document["partitions"][name]["blocks"], [TRAIN_DIR, TEST_DIR])
        for name in ("train", "test")
    }
    for name, part in blocks.items():
        rows = sum(int(b["n_rows"]) for b in part)
        print(f"{name:<6} {len(part):>2} files {rows:>12,} rows"
              + ("" if cap is None else f", reading the first {cap:,} of each"))
    print()

    started = time.time()
    X, y = {}, {}
    for name in ("train", "test"):
        X[name], y[name] = mo.load_partition(
            blocks[name],
            FEATURES,
            classes=CLASSES,
            max_rows_per_block=cap,
            progress=reader_progress(name),
        )
    read_seconds = time.time() - started

    print()
    print("The scaler sees the training side and nothing else. It is handed one array and")
    print("cannot reach a row that is not in it.")
    scaler = mo.fit_scaler(X["train"])
    for name in ("train", "test"):
        mo.transform_in_place(X[name], scaler)

    seen = int(np.ravel(scaler.n_samples_seen_)[0])
    print()
    print(f"rows read                    : train {len(X['train']):,}, test {len(X['test']):,}")
    print(f"rows the scaler saw          : {seen:,}")
    print(f"rows the training side holds : {len(X['train']):,}")
    print(f"time to read and scale       : {read_seconds:.1f}s read, "
          f"{time.time() - started - read_seconds:.1f}s scale")
    print()
    print(mo.scaler_statistics(scaler, FEATURES).head(8).to_string(index=False))

    counts = pd.DataFrame(
        {
            "class": CLASSES,
            "train": np.bincount(y["train"].astype("int64"), minlength=len(CLASSES)),
            "test": np.bincount(y["test"].astype("int64"), minlength=len(CLASSES)),
        }
    )
    print()
    print(counts.to_string(index=False))

    assert seen == len(X["train"]), "the scaler saw a number of rows the training side does not hold"
    assert (counts["train"] > 0).all() and (counts["test"] > 0).all(), (
        "a class is missing from one side of the distributed split, so it cannot be scored"
    )
    assert X["train"].shape[1] == len(FEATURES)
    assert np.isfinite(X["train"]).all() and np.isfinite(X["test"]).all(), (
        "scaling produced a value that is not finite, so something overflowed float32 and "
        "nothing below this point is a result"
    )

    return {
        "X_train": X["train"],
        "y_train": y["train"],
        "X_test": X["test"],
        "y_test": y["test"],
        "scaler": scaler,
        "counts": counts,
        "read_seconds": read_seconds,
    }

The second set of rows is the two-tier split, and it needs no reading at all: NB04 already
wrote it out, scaled by the scaler it fitted on its own training partition.

The distributed split divides eight of the nineteen classes by whole capture session and
the other eleven inside a single session, and it was not built to keep sessions apart. The
two-tier split was: for the eight classes recorded more than once, the sessions a model
trains on are not the sessions it is tested on. For the other eleven there is one recording
and no way to do that, so those rows are three consecutive stretches of one session and
their scores still carry whatever the session carries. The split is an improvement on eight
classes out of nineteen, not a fix.

The validation partition is loaded by nothing here. The published training procedure has no
validation split, and adding one to the run that reproduces it would be a second change.

In [ ]:
def load_ours(fast: bool) -> dict:
    section("Loading the two-tier split from the arrays NB04 wrote")
    mode = "fast" if fast else "full"
    array_dir = NB04_DIRS[mode]
    fell_back = not array_dir.exists()
    if fell_back:
        array_dir = NB04_DIRS["full"]

    X, y = {}, {}
    for name in ("train", "test"):
        path = array_dir / f"records_{name}.npz"
        if not path.exists():
            raise FileNotFoundError(f"{path} is missing. NB04 writes it; it has to be on Drive.")
        with np.load(path, allow_pickle=False) as npz:
            features = [str(v) for v in npz["features"]]
            classes = [str(v) for v in npz["classes"]]
            if features != FEATURES:
                raise ValueError(f"{path.name} holds different columns than the manifest lists")
            if classes != CLASSES:
                raise ValueError(f"{path.name} holds different classes than the manifest lists")
            X[name] = npz["X"]
            y[name] = npz["y"].astype("int64")
        print(f"  {path.name:<22} {str(X[name].shape):>20}   {X[name].dtype}")

    if fell_back and fast:
        print()
        print(f"{NB04_DIRS['fast']} is not on Drive, so the fast pass takes a stratified sample")
        print("of the full arrays instead. Same rows, fewer of them, same class proportions.")
        for name in ("train", "test"):
            index = rn.stratified_subsample(y[name], cap=FAST_ROW_CAPS[name], seed=SEED)
            X[name], y[name] = X[name][index], y[name][index]
            print(f"  {name:<6} cut to {len(index):,} rows")

    counts = pd.DataFrame(
        {
            "class": CLASSES,
            "train": np.bincount(y["train"], minlength=len(CLASSES)),
            "test": np.bincount(y["test"], minlength=len(CLASSES)),
        }
    )
    print()
    print(f"read from {array_dir}")
    print(f"train {len(y['train']):,} rows, test {len(y['test']):,} rows, "
          f"{len(FEATURES)} features, already scaled by NB04's scaler")
    print()
    print(counts.to_string(index=False))

    assert (counts["train"] > 0).all() and (counts["test"] > 0).all(), (
        "a class is missing from one side of the two-tier split, so it cannot be scored"
    )
    assert np.isfinite(X["train"]).all() and np.isfinite(X["test"]).all()

    return {
        "X_train": X["train"],
        "y_train": y["train"],
        "X_test": X["test"],
        "y_test": y["test"],
        "counts": counts,
        "array_dir": str(array_dir),
    }


LOADERS = {"shipped": load_shipped, "two_tier": load_ours}

Now the model, which is the same object in four of the seven runs.

The architecture, from the paper: the input is reshaped to (samples, features, 1), then a
1D convolution with 32 filters of kernel 3 and ReLU, max pooling over 2, a second 1D
convolution with 64 filters of kernel 3 and ReLU, max pooling over 2, flatten, a dense
layer of 128 with ReLU, and a dense softmax over the classes. Adam, categorical
cross-entropy, batch 32, ten epochs, no class weighting and no early stopping.

The reshape is worth stopping on. It puts each record on its own, with the feature list as
the axis the convolution slides along. The model therefore never sees two records at once,
and what its filters learn is a pattern across neighbouring columns of a single row, which
means across whichever columns the file happens to list next to each other. It is not
reading time. That is what the published model does and it is reproduced unchanged; the
version that does read time is NB06, and having this one to compare against is the reason
that comparison will mean anything.

The other three settings are left alone for the same reason. Nothing reweights the classes,
nothing stops training when the rare classes stop improving, and no rows are held back
during training to notice either. On a corpus with this imbalance those choices have an
obvious cost, and the per-class tables below are where it shows up.

The two configuration builders differ in one line. The run on the distributed split is the
root of the chain; the run on the two-tier split takes that root as its parent and changes
`split`, and nothing else.

In [ ]:
def published_config(task, mode, parent):
    classes = TASK_CLASSES[task]
    config = {
        "run_id": f"published_cnn_{task.replace('-', '')}",
        "parent": None if parent is None else parent["run_id"],
        "model": "mohammadi_cnn",
        "task": task,
        "split": "shipped",
        "input": "one record, reshaped to (features, 1)",
        "n_features": len(FEATURES),
        "preprocessing": "StandardScaler fitted on the training side of the split, and nothing else",
        "optimizer": mo.COMPILE["optimizer"],
        "loss": mo.COMPILE["loss"],
        "batch_size": mo.FIT["batch_size"],
        "epochs": mo.FIT["epochs"],
        "class_weight": mo.FIT["class_weight"],
        "early_stopping": mo.FIT["early_stopping"],
        "seed": SEED,
        "observed": {
            "mode": mode,
            "n_classes": len(classes),
            "classes": classes,
            "features": FEATURES,
            "scaler_fitted_by": "baselines/mohammadi/data.py, on the rows this notebook read",
            "model": mo.describe(len(FEATURES), len(classes)),
            "notebook": NOTEBOOK,
            "git_sha": GIT_SHA,
            "run_date": RUN_DATE,
        },
    }
    config["observed"]["changed_from_parent"] = sorted(rn.assert_single_change(config, parent))
    return config


def ours_config(task, mode, parent):
    config = dict(published_config(task, mode, None))
    config["run_id"] = f"ours_cnn_{task.replace('-', '')}"
    config["split"] = "two_tier"
    config["parent"] = parent["run_id"]
    config["observed"] = dict(config["observed"])
    config["observed"]["scaler_fitted_by"] = (
        "src/preprocessing.py in NB04, on the two-tier training partition"
    )
    config["observed"]["changed_from_parent"] = sorted(rn.assert_single_change(config, parent))
    return config


def train_cnn(step, r):
    """Fit and save in one statement, so a dropped session cannot lose the fit."""
    task = step["task"]
    classes = TASK_CLASSES[task]
    data = r["data"][step["data"]]
    config = step["config"]
    config["observed"]["n_train"] = int(len(data["y_train"]))
    config["observed"]["n_test"] = int(len(data["y_test"]))
    return rn.fit_and_save_keras(
        r["out_dir"],
        config["run_id"],
        build=lambda: mo.build_model(len(FEATURES), len(classes)),
        fit=lambda model, X, y, callbacks: mo.fit(
            model, X, y, callbacks, n_classes=len(classes), verbose=FIT_VERBOSE
        ),
        X_train=mo.reshape(data["X_train"]),
        y_train=codes_for(task, data["y_train"]),
        X_test=mo.reshape(data["X_test"]),
        y_test=codes_for(task, data["y_test"]),
        classes=classes,
        config=config,
        parent=step["parent"],
        seed=SEED,
        predict_batch_size=PREDICT_BATCH,
    )

The third model is a random forest on single records.

Both of the others are the same network, so if that architecture is a bad fit for these
rows the two of them will agree about it and neither will say so. A forest on the same rows
is the check on that. It sees one record at a time, the same forty-four columns, the same
two-tier split, and it makes no assumption at all about the columns being ordered
meaningfully, which is the assumption the convolution is built on.

The folds come first. Five stratified folds are cut inside the training partition, each one
fitted and scored on the part of training it did not see, which says how much the score
moves when the training rows change. The test partition is not involved in any of that and
is evaluated once, at the end, by a forest fitted on the whole training partition. That is
the rule the whole project runs on: tuning happens on training and validation, and the test
set is scored once per configuration.

Two practical limits are recorded rather than hidden. The forest trains on a stratified
sample of the training partition rather than all six million rows, because six fits of a
hundred trees on six million rows does not finish in a session, and the leaf size is held at
twenty for the same reason. Both are in the config file the run writes, and both apply
equally to the nineteen-class and six-class runs, so the comparison between those two is
still a comparison of one change.

In [ ]:
from sklearn.ensemble import RandomForestClassifier


def forest_config(task, mode, cap, parent):
    classes = TASK_CLASSES[task]
    config = {
        "run_id": f"forest_{task.replace('-', '')}",
        "parent": None if parent is None else parent["run_id"],
        "model": "random_forest",
        "task": task,
        "split": "two_tier",
        "input": "one record, 44 features",
        "n_features": len(FEATURES),
        "preprocessing": "StandardScaler fitted on the training side of the split, and nothing else",
        "n_estimators": FOREST["n_estimators"],
        "min_samples_leaf": FOREST["min_samples_leaf"],
        "max_features": FOREST["max_features"],
        "k_folds": K_FOLDS,
        "train_rows_cap": cap,
        "seed": SEED,
        "observed": {
            "mode": mode,
            "n_classes": len(classes),
            "classes": classes,
            "features": FEATURES,
            "notebook": NOTEBOOK,
            "git_sha": GIT_SHA,
            "run_date": RUN_DATE,
            "why_capped": (
                "five folds plus a final fit is six forests, and six forests of "
                f"{FOREST['n_estimators']} trees on the whole training partition does not "
                "finish in a Colab session"
            ),
        },
    }
    config["observed"]["changed_from_parent"] = sorted(rn.assert_single_change(config, parent))
    return config


def forest_sample(data, cap):
    """The rows the forest trains on, drawn once and reused by both forest runs."""
    if "forest_index" in data:
        return data["forest_index"]

    index = rn.stratified_subsample(data["y_train"], cap=cap, seed=SEED)
    kept = pd.DataFrame(
        {
            "class": CLASSES,
            "in_training": np.bincount(data["y_train"], minlength=len(CLASSES)),
            "sampled": np.bincount(data["y_train"][index], minlength=len(CLASSES)),
        }
    )
    kept["share"] = kept["sampled"] / kept["in_training"].clip(lower=1)
    print(f"training partition {len(data['y_train']):,} rows, sampled down to {len(index):,} "
          f"at a cap of {cap:,}")
    print()
    print(kept.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print()
    print("Every class keeps the same share of the sample that it had of the partition, so")
    print("the imbalance the model is asked to deal with is the imbalance the corpus has.")
    print()

    data["forest_index"] = index
    data["forest_sample"] = kept
    return index


def train_forest(step, r):
    """Cross-validate, fit, score and save in one statement."""
    task = step["task"]
    classes = TASK_CLASSES[task]
    data = r["data"][step["data"]]
    config = step["config"]
    index = forest_sample(data, config["train_rows_cap"])
    config["observed"]["n_train"] = int(len(index))
    config["observed"]["n_test"] = int(len(data["y_test"]))
    return rn.kfold_fit_and_save(
        r["out_dir"],
        config["run_id"],
        build=lambda: RandomForestClassifier(
            n_estimators=FOREST["n_estimators"],
            min_samples_leaf=FOREST["min_samples_leaf"],
            max_features=FOREST["max_features"],
            n_jobs=-1,
            random_state=SEED,
        ),
        X_train=data["X_train"][index],
        y_train=labels_for(task, data["y_train"][index]),
        X_test=data["X_test"],
        y_test=labels_for(task, data["y_test"]),
        classes=classes,
        config=config,
        parent=step["parent"],
        n_splits=K_FOLDS,
        seed=SEED,
        features=FEATURES,
    )


TRAINERS = {"cnn": train_cnn, "forest": train_forest}

The order the seven run in is a decision, not an accident, and it is not the order they
were described in.

A full pass is hours long and a Colab session can end inside one, so the question that
decides the order is: if it stops after two runs, or after three, which two or three would
I rather have. The answer is the two nineteen-class convolutional runs, because the
comparison between them is the baseline H1 is measured against and it is the one result
this notebook exists to produce. They go first, the two-tier one before the published one,
so that the model every later notebook is compared against is the very first thing on disk.
The forest at nineteen classes goes third, which is the point at which the nineteen-class
picture is complete and can be read on its own.

Then the same three at six classes, in the same order, which gives the family-level version
of the same comparison. The two-class reproduction goes last. It is the least informative
run in the notebook, and a two-class score of 0.99 on a corpus that is 97% attack traffic
tells almost nothing, but the paper reports it so it is reproduced.

Every configuration is built here, before anything runs, so the parent of a run is
available whether or not the parent has been trained yet. The nineteen-class published
configuration is the parent of the two-tier run that executes before it, and the check that
they differ in one key does not care which of them was fitted first.

The cost of interleaving the two splits is that both sets of rows are in memory at the same
time, a little under three gigabytes. A set is released as soon as no run left in the plan
needs it.

In [ ]:
RUN_ORDER = [
    ("ours_cnn_19class", "cnn", "two_tier", "19-class",
     "The nineteen-class network on the split that holds recording sessions apart. This is "
     "the model every later notebook is measured against, and it is on disk first for that "
     "reason."),
    ("published_cnn_19class", "cnn", "shipped", "19-class",
     "Both nineteen-class convolutional runs. The split comparison the first hypothesis "
     "needs is complete, so a session that ends here has still produced the main result of "
     "this notebook."),
    ("forest_19class", "forest", "two_tier", "19-class",
     "A nineteen-class result that does not depend on the convolution being the right idea, "
     "so the two scores above can be read against something outside their own family. The "
     "nineteen-class picture is complete."),
    ("ours_cnn_6class", "cnn", "two_tier", "6-class",
     "The six-class task on the two-tier split, which is where the family-level scores "
     "start."),
    ("published_cnn_6class", "cnn", "shipped", "6-class",
     "Both six-class convolutional runs, so the split comparison now exists at family level "
     "as well as per class."),
    ("forest_6class", "forest", "two_tier", "6-class",
     "The forest at family level. Every run that feeds a per-class comparison is now on "
     "disk, and only the two-class reproduction is left."),
    ("published_cnn_2class", "cnn", "shipped", "2-class",
     "The two-class reproduction, which completes the set of numbers the paper reports."),
]


def build_plan(mode: str) -> list:
    """Every configuration, built before anything trains, in the order they will run."""
    cap = FOREST_TRAIN_ROWS[mode]
    configs = {}
    configs["published_cnn_19class"] = published_config("19-class", mode, None)
    configs["published_cnn_6class"] = published_config(
        "6-class", mode, configs["published_cnn_19class"]
    )
    configs["published_cnn_2class"] = published_config(
        "2-class", mode, configs["published_cnn_19class"]
    )
    configs["ours_cnn_19class"] = ours_config("19-class", mode, configs["published_cnn_19class"])
    configs["ours_cnn_6class"] = ours_config("6-class", mode, configs["ours_cnn_19class"])
    configs["forest_19class"] = forest_config("19-class", mode, cap, None)
    configs["forest_6class"] = forest_config("6-class", mode, cap, configs["forest_19class"])

    plan = []
    for run_id, kind, data, task, note in RUN_ORDER:
        config = configs[run_id]
        plan.append(
            {
                "run_id": run_id,
                "kind": kind,
                "data": data,
                "task": task,
                "banked": note,
                "config": config,
                "parent": configs.get(config["parent"]),
            }
        )

    if sorted(s["run_id"] for s in plan) != sorted(configs):
        raise ValueError("the order and the configurations do not name the same seven runs")
    return plan


PLAN = build_plan("full")
plan_table = pd.DataFrame(
    [
        {
            "order": i,
            "run_id": s["run_id"],
            "model": s["config"]["model"],
            "task": s["task"],
            "rows_from": s["data"],
            "parent": s["config"]["parent"] or "-",
            "the_one_change": ", ".join(s["config"]["observed"]["changed_from_parent"]) or "root",
        }
        for i, s in enumerate(PLAN, start=1)
    ]
)
print(plan_table.to_string(index=False))
print()
print("The parent of a run is a configuration, not a result, so a run can be the parent of")
print("something that trains before it does.")

When all seven have run, or all of the ones that are going to, they go on one table.

Underneath it the two convolutional runs of each task are put against each other, which is
the comparison the ordering was built around. They are the same network with the same
settings on the same columns, so the only thing that can account for a difference between
them is which rows were on which side.

The column that matters on the wide table is the last one, the distance between weighted F1
and macro F1.
Weighted F1 averages the per-class scores by how many rows each class has, so it is
dominated by the handful of flood classes that hold most of the corpus. Macro F1 averages
them evenly, so a class with nine hundred rows counts as much as a class with two million.
When those two numbers are close, the model is doing roughly as well on the rare classes as
on the common ones. When they are far apart, the rare classes are failing and the weighted
figure is hiding it.

In [ ]:
def compare(r) -> dict:
    section("Every run side by side")
    table = rn.comparison_frame(r["runs"])
    print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print()

    cnn = {
        (run["config"]["task"], run["config"]["split"]): run["metrics"]["macro_f1"]
        for run in r["runs"]
        if run["config"]["model"] == "mohammadi_cnn"
    }
    paired = [
        {
            "task": task,
            "published_shipped": cnn[(task, "shipped")],
            "same_model_two_tier": cnn[(task, "two_tier")],
            "difference": cnn[(task, "two_tier")] - cnn[(task, "shipped")],
        }
        for task in TASKS
        if (task, "shipped") in cnn and (task, "two_tier") in cnn
    ]
    split_table = pd.DataFrame(paired)
    for column in [c for c in split_table.columns if c != "task"]:
        if not pd.api.types.is_numeric_dtype(split_table[column]):
            raise TypeError(f"{column} came out as {split_table[column].dtype}, not numeric")
    if len(split_table):
        print("The same network, the same settings, the same columns, a different split.")
        print("Macro F1 on each side:")
        print()
        print(f"{'task':<10}{'published, shipped':>22}{'same model, two-tier':>24}"
              f"{'difference':>14}")
        for row in split_table.itertuples():
            print(f"{row.task:<10}{row.published_shipped:>22.4f}"
                  f"{row.same_model_two_tier:>24.4f}{row.difference:>+14.4f}")
        print()
        print("The difference is the split and nothing else, which is what makes it readable.")
        print("Eight of the nineteen classes are tested on recording sessions the model has not")
        print("seen; the other eleven are tested on the tail of the session they trained on.")
        print()

    print("What the gap says, run by run:")
    for row in table.itertuples():
        if row.gap < -0.02:
            reading = ("macro sits above weighted, so it is the large classes that are behind "
                       "and the small ones carrying the average")
        elif abs(row.gap) < 0.02:
            reading = "the rare classes score about as well as the common ones"
        elif row.gap < 0.10:
            reading = "a few classes are behind the rest"
        elif row.gap < 0.25:
            reading = "several classes are scoring far below what the headline suggests"
        else:
            reading = "the headline is the large classes; some classes are barely detected"
        print(f"  {row.run_id:<24} accuracy {row.accuracy:.4f}, weighted F1 {row.weighted_f1:.4f}, "
              f"macro F1 {row.macro_f1:.4f}, gap {row.gap:.4f}")
        print(f"  {'':<24} {reading}")
    print()
    worst = table.loc[table["gap"].idxmax()]
    print(f"Widest gap: {worst['run_id']}, {worst['gap']:.4f}. Accuracy reads "
          f"{worst['accuracy']:.4f} on the same predictions.")
    print("An accuracy near one on this corpus is close to what always answering the largest")
    print("class would give, so on its own it says almost nothing about whether an attack")
    print("class is detected.")
    return {"comparison": table, "split_comparison": split_table}

Then the same thing again without any averaging.

Every model's per-class F1 on the nineteen-class task, one column per run, sorted so the
classes that fail are at the top. This is the table the rest of the project works from: a
class sitting at zero here is a class no baseline detects, and an improvement claimed later
has to show up on one of these rows rather than in a mean.

The two support columns are separate because the two splits have different test partitions,
so a class has a different number of rows to be scored on depending on which split a run
used.

In [ ]:
def per_class(r) -> dict:
    section("Per-class F1 on the 19-class task, every model")
    table = rn.per_class_matrix(r["runs"], task="19-class")
    print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    score_columns = [c for c in table.columns if c != "label" and not c.startswith("n_test_")]
    failed = table[table[score_columns].max(axis=1) < 0.50]
    print()
    print(f"classes no model reaches F1 0.50 on: {len(failed)} of {len(table)}")
    if len(failed):
        print("  " + ", ".join(failed["label"].tolist()))
    zero = table[table[score_columns].max(axis=1) == 0.0]
    print(f"classes every model scores 0.00 on : {len(zero)}")
    if len(zero):
        print("  " + ", ".join(zero["label"].tolist()))
    return {"per_class": table}

The driver walks the plan in order and does three things at each step.

It looks for a `metrics.json` in the run's folder first. If there is one, that run finished
on some earlier attempt, and it is read back off disk rather than fitted again. What comes
back has the same shape as a run just produced, so the tables at the end do not know the
difference.

If there is no `metrics.json`, it makes sure the rows that run needs are loaded, which is
where a set of rows gets read for the first time, and fits. A run whose data has already
been loaded by an earlier run in the plan costs nothing extra. If every run that needs the
capture files turns out to be complete already, the capture files are never opened.

Then it says what is banked, and releases any set of rows that no remaining run needs.

The only thing the argument changes is how many rows get read: a cap on rows per capture
file for the distributed split, the smaller arrays from NB04's own fast pass for the
two-tier side, and a smaller sample for the forest. Every model, every setting and every
step is the same in both passes.

In [ ]:
def run_baselines(fast: bool) -> dict:
    mode = "fast" if fast else "full"
    out_dir = OUT_DIRS[mode]
    out_dir.mkdir(parents=True, exist_ok=True)
    plan = build_plan(mode)

    started = time.time()
    r = {"mode": mode, "fast": fast, "out_dir": out_dir, "runs": [], "data": {}, "skipped": []}

    for i, step in enumerate(plan, start=1):
        section(f"Run {i} of {len(plan)} - {step['run_id']}")
        existing = rn.load_run(out_dir, step["run_id"]) if RESUME else None

        if existing is not None:
            print(f"{step['run_id']}: already complete, skipping")
            print(f"  read back from {existing['run_dir']}, macro F1 "
                  f"{existing['metrics']['macro_f1']:.4f}")
            r["skipped"].append(step["run_id"])
            run = existing
        else:
            if step["data"] not in r["data"]:
                r["data"][step["data"]] = LOADERS[step["data"]](fast)
            run = TRAINERS[step["kind"]](step, r)
            print()
            report(run)

        r["runs"].append(run)
        banked(r["runs"], plan, i)

        still_needed = {s["data"] for s in plan[i:]}
        for name in [n for n in list(r["data"]) if n not in still_needed]:
            r["data"].pop(name)
            gc.collect()
            print(f"  the {name} rows are released; no run left in the plan needs them")
            print()

    r["data"].clear()
    gc.collect()

    merge(r, compare(r))
    merge(r, per_class(r))

    r["elapsed_s"] = time.time() - started
    section(f"What the {mode} pass has in {out_dir}")
    for run in r["runs"]:
        files = sorted(p.name for p in Path(run["run_dir"]).iterdir())
        fitted = "read back" if run["name"] in r["skipped"] else "fitted here"
        print(f"  {run['name']:<24} {fitted:<12} {', '.join(files)}")
    print(f"  {len(r['runs'])} runs, {len(r['skipped'])} of them already complete, "
          f"{r['elapsed_s'] / 60:.1f} minutes")
    return r

Before starting it, how long it will take.

The arithmetic is not a measurement. It counts the gradient steps the full pass will
actually take, which is exact, and divides by a guessed rate, which is not. The rate is
written in the cell and can be changed once a real run has been timed. Its only job is to
say whether this is a twenty minute job or most of a day, and it is here so that the
decision to stop and come back with a longer session gets made now rather than four hours
in.

Runs that are already on disk are counted at zero, because the driver will read them back
rather than fit them. So a second look at this cell after a session that got halfway
through shows what is actually left.

In [ ]:
STEP_RATE = 700
FOREST_ROWS_PER_SECOND = 20_000
CSV_ROWS_PER_SECOND = 120_000


def planned_rows(mode: str) -> dict:
    """How many rows each run will train and test on, without reading any of them."""
    cap = ROWS_PER_FILE[mode]
    shipped = {
        name: sum(
            min(int(b["n_rows"]), cap) if cap else int(b["n_rows"])
            for b in SPLITS["shipped"]["partitions"][name]["blocks"]
        )
        for name in ("train", "test")
    }

    array_dir = NB04_DIRS[mode] if NB04_DIRS[mode].exists() else NB04_DIRS["full"]
    fell_back = array_dir != NB04_DIRS[mode]
    two_tier = {}
    for name in ("train", "test"):
        with np.load(array_dir / f"records_{name}.npz", allow_pickle=False) as npz:
            n_rows = int(npz["y"].shape[0])
        if fell_back and mode == "fast":
            n_rows = min(n_rows, FAST_ROW_CAPS[name])
        two_tier[name] = n_rows

    return {"shipped": shipped, "two_tier": two_tier}


def estimate(mode: str) -> pd.DataFrame:
    rows = planned_rows(mode)
    out_dir = OUT_DIRS[mode]
    lines = []
    for i, step in enumerate(build_plan(mode), start=1):
        n_train = rows[step["data"]]["train"]
        done = RESUME and rn.load_run(out_dir, step["run_id"]) is not None
        if step["kind"] == "cnn":
            steps = int(np.ceil(n_train / BATCH_SIZE)) * EPOCHS
            seconds = steps / STEP_RATE
        else:
            n_train = min(n_train, FOREST_TRAIN_ROWS[mode])
            steps = 0
            seconds = (K_FOLDS + 1) * n_train / FOREST_ROWS_PER_SECOND
        lines.append(
            {
                "order": i,
                "run_id": step["run_id"],
                "train_rows": n_train,
                "gradient_steps": steps,
                "minutes": 0.0 if done else seconds / 60,
                "status": "already complete" if done else "to run",
            }
        )

    table = pd.DataFrame(lines)
    table["cumulative_minutes"] = table["minutes"].cumsum()
    for column in ("train_rows", "gradient_steps", "minutes", "cumulative_minutes"):
        if not pd.api.types.is_numeric_dtype(table[column]):
            raise TypeError(f"{column} came out as {table[column].dtype}, not numeric")

    to_read = {
        step["data"]
        for step in build_plan(mode)
        if not (RESUME and rn.load_run(out_dir, step["run_id"]) is not None)
    }
    read_minutes = (
        (rows["shipped"]["train"] + rows["shipped"]["test"]) / CSV_ROWS_PER_SECOND / 60
        if "shipped" in to_read
        else 0.0
    )

    print(f"{mode} pass")
    print(table.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
    print()
    print("  A forest takes no gradient steps. Its minutes are six fits, five folds and one")
    print("  final, over the rows left after the cap.")
    print(f"  gradient steps still to take : {table.loc[table['status'] == 'to run', 'gradient_steps'].sum():,}")
    print(f"  reading the capture files    : {read_minutes:,.1f} minutes")
    print(f"  fitting                      : {table['minutes'].sum():,.1f} minutes")
    print(f"  total                        : {read_minutes + table['minutes'].sum():,.1f} minutes "
          f"({(read_minutes + table['minutes'].sum()) / 60:,.1f} hours)")
    return table


print(f"assumed rates: {STEP_RATE:,} gradient steps a second, {FOREST_ROWS_PER_SECOND:,} forest "
      f"rows a second, {CSV_ROWS_PER_SECOND:,} csv rows a second.")
print("Guesses, not measurements. The step counts are exact.")
print()
ESTIMATE = {mode: estimate(mode) for mode in ("fast", "full")}
print()
print("If the full pass is longer than the session available, stop here. The driver reads")
print("finished runs back off disk, so two short sessions cost the same as one long one plus")
print("the time to load the rows again.")

Both passes run here, fast first. If the fast one raises, the full one never starts, which
is the point: a wrong path, a shape that does not line up or a configuration that would
confound two changes costs a few minutes rather than most of a day.

The fast pass resumes in the same way the full one does, so a second run of this cell will
skip it entirely. Deleting the `NB05_fast` folder forces the plumbing check to happen
again.

In [ ]:
BANNERS = {
    "fast": [
        "FAST PASS",
        "every model, every task, every check, on a fraction of the rows",
        "for shapes and plumbing only",
        "not a result, and never entered in the ledger",
    ],
    "full": [
        "FULL PASS",
        "every row of both splits, ten epochs at batch 32",
        "this is the pass that goes in the ledger",
    ],
}

results = {}
for fast in (True, False):
    name = "fast" if fast else "full"
    banner(BANNERS[name])
    results[name] = run_baselines(fast)

FAST, FULL = results["fast"], results["full"]
banner([f"fast pass {FAST['elapsed_s'] / 60:.1f} min, full pass {FULL['elapsed_s'] / 60:.1f} min",
        f"{len(FULL['runs'])} runs in {FULL['out_dir']}",
        f"{len(FULL['skipped'])} were already complete and were read back rather than refitted"])

The comparison table once more, then one ledger entry per run, ready to paste into
`RESULTS_LEDGER.md`. Colab cannot push to the repository from a cell, so until the
artefacts are moved across by hand the saved copy of this notebook is the only durable
record of what the run produced.

In [ ]:
print("=" * 100)
print("NB05 full pass - every run, in the order they were written")
print("=" * 100)
print(FULL["comparison"].to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print("=" * 100)
print("NB05 full pass - per-class F1, 19-class task")
print("=" * 100)
print(FULL["per_class"].to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()

STATUS = "reference run" + (", working tree dirty" if GIT_DIRTY else "")

overall = f"""
### NB05 — baseline models ({RUN_DATE})

| field | value |
|---|---|
| notebook | {NOTEBOOK} |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| pass reported | full |
| runs | {len(FULL["runs"])}, of which {len(FULL["skipped"])} were read back from an earlier session |
| runtime | fast {FAST["elapsed_s"] / 60:.1f} min, full {FULL["elapsed_s"] / 60:.1f} min |
| features | {len(FEATURES)}, after dropping {", ".join(MANIFEST["columns"]["dropped"])} |
| artifacts | {FULL["out_dir"]} |
| status | {STATUS} |

| run | model | task | split | accuracy | weighted F1 | macro F1 | gap |
|---|---|---|---|---|---|---|---|
""" + "\n".join(
    f"| {row.run_id} | {row.model} | {row.task} | {row.split} | {row.accuracy:.4f} | "
    f"{row.weighted_f1:.4f} | {row.macro_f1:.4f} | {row.gap:.4f} |"
    for row in FULL["comparison"].itertuples()
)

blocks = [overall]
for position, run in enumerate(FULL["runs"], start=1):
    config, metrics = run["config"], run["metrics"]
    changed = config["observed"]["changed_from_parent"]
    zero = sorted(label for label, value in metrics["per_class_f1"].items() if value == 0.0)
    weakest = sorted(metrics["per_class_f1"].items(), key=lambda pair: pair[1])[:4]
    folds = metrics.get("cross_validation")
    blocks.append(f"""
### NB05 — {config["run_id"]} ({RUN_DATE})

| field | value |
|---|---|
| notebook | {NOTEBOOK} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {config["seed"]} |
| written | {position} of {len(FULL["runs"])} |
| model | {config["model"]} |
| task | {config["task"]}, {metrics["n_classes"]} classes |
| split | {config["split"]} |
| parent | {config["parent"] or "none, this is where the chain starts"} |
| the one change | {", ".join(changed) if changed else "none, root configuration"} |
| training rows | {metrics["n_train"]:,} |
| test rows | {metrics["n_test"]:,} |
| accuracy | {metrics["accuracy"]:.4f} (chance {metrics["chance_rate"]:.4f}, largest class {metrics["majority_class_rate"]:.4f}) |
| weighted P / R / F1 | {metrics["weighted_precision"]:.4f} / {metrics["weighted_recall"]:.4f} / {metrics["weighted_f1"]:.4f} |
| macro P / R / F1 | {metrics["macro_precision"]:.4f} / {metrics["macro_recall"]:.4f} / {metrics["macro_f1"]:.4f} |
| weighted F1 minus macro F1 | {metrics["weighted_f1"] - metrics["macro_f1"]:.4f} |
| cross-validation | {"none" if not folds else f"{folds['n_splits']}-fold on training, macro-F1 {folds['macro_f1_mean']:.4f} +/- {folds['macro_f1_sd']:.4f}"} |
| classes at F1 0.00 | {", ".join(zero) if zero else "none"} |
| four weakest classes | {", ".join(f"{label} {value:.2f}" for label, value in weakest)} |
| train seconds | {metrics["train_seconds"]:,.1f} |
| inference seconds | {metrics["inference_seconds"]:,.1f} ({metrics["inference_rows_per_second"]:,.0f} rows/s) |
| artifacts | {run["run_dir"]} |
| status | {STATUS} |
""")

print("=" * 100)
print("paste into RESULTS_LEDGER.md")
print("=" * 100)
print("\n".join(blocks))